# Exercise 0 - PydanticAI and OpenRouter

In this exercise, you get to work with data validation using Pydantic v2 and PydanticAI. Output in general from LLMs are unstructured and hard to work with, to get any value from it you need to structure the output and validate its contents. This can be done with Pydantic.

## 0. Summarize job ads

In the data folder, there are a few job ads taken from arbetsförmedlingen.se.

### a) Read all the jobs ads into python



In [1]:
from pathlib import Path


def load_ads(data_dir: Path) -> dict[str, str]:
    """Read every job ad (ads*.txt) in a folder.

    Args:
        data_dir: Folder that contains the ads*.txt files.

    Returns:
        A dict mapping the file name without extension (e.g. "ads1")
        to the full text of that job ad.
    """
    ads = {}  # filled below: {"ads1": "<text>", "ads2": "<text>", ...}

    # sorted() keeps the order the same on every run (ads1, ads2, ads3)
    for file in sorted(data_dir.glob("ads*.txt")):
        # file.stem is the name without ".txt", so "ads1.txt" becomes "ads1"
        # encoding="utf-8" avoids garbled characters on Windows
        ads[file.stem] = file.read_text(encoding="utf-8")

    return ads


# ".." = go up one folder, from exercise_0/ to exercises/, then into data/
ads = load_ads(Path("..") / "data")

# Quick check that the files were read
for name, text in ads.items():
    print(name, "->", len(text), "characters")

ads1 -> 5519 characters
ads2 -> 7496 characters
ads3 -> 3968 characters
ads4 -> 3114 characters


### b) Create a function that uses PydanticAI agent to summarize a job ad. This function should take in an ad as its parameter and return a summary.

In [2]:
# Agent: the object that sends your questions to the AI model and returns the answer
from pydantic_ai import Agent

# OpenAIChatModel: a wrapper that talks to any service using OpenAI's chat format
from pydantic_ai.models.openai import OpenAIChatModel

# OpenAIProvider: holds the connection details (where to connect and how to log in)
from pydantic_ai.providers.openai import OpenAIProvider

import os  # lets us read environment variables

# It searches upward from the current folder, so it finds the .env in your project root.
# reads key=value pairs from a .env file
from dotenv import load_dotenv  


# NVIDIA's API speaks the same "chat completions" language as OpenAI,
# so we can reuse the OpenAI classes and just point them at NVIDIA's address.
model = OpenAIChatModel(
    "nvidia/nemotron-3.5-lightning-30b-a3b",  # the model name, same as lecture 01
    provider = OpenAIProvider(
        base_url="https://integrate.api.nvidia.com/v1",  # where to send requests
        api_key = os.environ["NVIDIA_API"], # key, read from .env (never hard-code it)
    ),
)

In [3]:
# The agent is the object that sends your prompt to the model and hands back the answer.
# `instructions` is the system prompt: it tells the model its job, every time it is called.
summary_agent = Agent(
    model,
    instructions = (
        "You are an assistant that summarizes job ads. "
        "Write a short clear summary in English covering the role, "
        "the main responsibilities and the key requirements."
    ),
)

async def summarize_ad(ad:str) -> str:
    """Summarize a single job ad.

    Args:
        ad: The full text of the job ad.

    Returns:
        A short summary written by the LLM.
    """
    # agent.run() sends the ad to the model as the user prompt and waits for the reply.
    # It is async, so we must `await` it. That works directly in a notebook cell.

    result = await summary_agent.run(ad, model_settings = {"timeout" : 120})

    # result.output is the model's answer in plain str
    return result.output

# Try in the first ad
summary = await summarize_ad(ads["ads1"])
print(summary)

                 pydantic-ai v2.62.0 • pydantic-ai-harness v0.62.0 • Python 3.13.7

      / \        agent: summary_agent • model: openai:nvidia/nemotron-3.5-lightning-30b-a3b
     /   \         tools: 0 • capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


**Role:**  
Data Engineer & Analytics Engineer (hybrid) shaping Instabee’s data platform on GCP and Snowflake to serve stakeholders, BI, data science, and product teams.

**Main Responsibilities:**  
- Design integration patterns and build/migrate source connectors (APIs, MongoDB, MySQL)  
- Set up and maintain data platform infrastructure, CI/CD pipelines, and IaC  
- Orchestrate data pipelines (Airflow) and model data using DBT/SQL with Python  
- Introduce data product principles (discoverability, trustworthiness) and support analytical stakeholder projects  

**Key Requirements:**  
- Proven experience with modern data stack (GCP, Snowflake, DBT, Python, FiveTran) and data modeling  
- Strong background in integrating diverse data sources and setting up CI/CD/IaC  
- University degree in a technical/analytical field + experience spanning both Data Engineering and Analytics Engineering  
- Agile mindset, stakeholder collaboration, and based in Stockholm (hybrid, 3 days/office)


- Agent(model, instructions=...): The agent wraps the model and adds the system prompt. Think of the system prompt as standing orders that apply to every call. The ad you pass in is the user prompt, the part that changes each time.

- async def and await: Talking to an LLM means waiting on the network. agent.run is async, so the function that calls it must also be async def and use await. In a normal .py script you would wrap the call in asyncio.run(...). In a notebook you can await directly. agent.run_sync also exists, but it often clashes with the notebook’s own event loop, so I’m avoiding it.

### c) Now create and export markdown files for each job ad and its corresponding summary.

In [4]:
# Folder for the generated files, next to the notebook: exercises/exercise_0/summaries/
output_dir = Path("summaries")

# parents/exist_ok: create the folder if missing, and don't error if it already exists
output_dir.mkdir(exist_ok=True)

async def export_summaries(ads:dict[str,str], output_dir:Path) -> None:
    """Summarize every ad and save ad + summary as one markdown file per ad.

    Args:
        ads: Mapping of ad name (e.g. "ads1") to the full ad text.
        output_dir: Folder where the .md files are written.
    """
    for name, ad in ads.items():
        # One LLM call per ad
        summary = await summarize_ad(ad)

        # Build the markdown text. "##" makes a heading.
        # The ad is placed under its own heading, followed by its summary.
        markdown = f"# {name}\n\n## Summary\n\n{summary}\n\n## Original ad\n\n{ad}\n"

        # Write it to e.g. summaries/ads1.md
        (output_dir / f"{name}.md").write_text(markdown, encoding="utf-8")
        print(f"Saved {name}.md")
await export_summaries(ads, output_dir)

Saved ads1.md
Saved ads2.md
Saved ads3.md
Saved ads4.md


- write_text(...) is the reverse of the read_text you used in 0a. It creates the file, or overwrites it if it exists. Use encoding="utf-8" here too, for the same reason as before.

- output_dir / f"{name}.md" uses the same / path-joining as before. The file name comes from the dict key, which is why we kept it.

- Sequential loop: The ads are summarized one after another, so three calls take roughly three times as long as one. That’s fine for three ads. If you had hundreds, you would run the calls concurrently with asyncio.gather

### d) Try to take other job ads from arbetsförmedlingen.se and see how well your function performs. (See ads4.txt)
- Go to arbetsförmedlingen and copy one job ad.

### e) Make the output structured using a pydantic model together with PydanticAI agent. The output from the agent should have the following fields:

- job_title
- description
- summary
- responsibilities
- words_in_article


In [5]:
from pydantic import BaseModel, Field

class JobAdSummary(BaseModel):
    """Structured summary of a job ad"""

    # Field(description=...) is not just documentation: PydanticAI sends these
    # descriptions to the LLM, so they act as per-field instructions.

    job_title: str = Field(description = "The title of the position")
    description: str = Field(description = "1-2 sentences about the company and the role")
    summary: str = Field("A short summary of the whole ad in English")
    responsibilities: str = Field("The main responsibilities, one short item per entry")
    words_in_article: str = Field("Number of words in the original job ad")

- BaseModel: Inheriting from it turns a plain class into a Pydantic model. Each line name: type declares a field and the type it must have.

- Types are rules: str means text, int means a whole number, and list[str] means a list of strings. If the LLM returns "about 300" for words_in_article, Pydantic rejects it. PydanticAI then sends the error back to the model and asks it to retry. That is the validation your exercise intro talks about.

- Field(description=...): Adds extra info to a field. Later you’ll also use it for constraints such as minimum and maximum values

In [6]:
extract_agent = Agent(
    model,
    output_type=JobAdSummary,  # the key change: the answer must fit this model
    instructions=(
        "You are an assistant that analyzes job ads. "
        "Fill in every field based only on the job ad. "
        "Write in English, even if parts of the ad are in Swedish."
    ),
)


async def extract_job_ad(ad: str) -> JobAdSummary:
    """Analyze a job ad and return it as a validated JobAdSummary.

    Args:
        ad: The full text of the job ad.

    Returns:
        A JobAdSummary object with all fields filled in and type-checked.
    """
    result = await extract_agent.run(ad, model_settings={"timeout": 120})
    # With output_type set, result.output is a JobAdSummary object, not a string
    return result.output


ad_info = await extract_job_ad(ads["ads4"])
print(ad_info.job_title)
print(ad_info.responsibilities)
print(ad_info.model_dump())  # the whole thing as a plain dict

Data Engineer
Design and operate data infrastructure (streaming and batch) and the systems that move data reliably at scale. Contribute to how events are defined, structured, and evolved across the platform. Ensure data quality, consistency, and observability across event pipelines. Ensure GDPR and PII compliance across data flows at scale. Contribute to data and architecture best practices that may span across the whole organisation.
{'job_title': 'Data Engineer', 'description': 'Mentimeter is seeking a Data Engineer to design and operate modern event‑driven data infrastructure, ensuring reliable data pipelines, quality, and GDPR/PI compliance across the company.', 'summary': 'The role involves building and improving a hybrid streaming and batch data platform that powers analytics, growth, and personalization, with a focus on data quality, observability, and scalability in a hybrid Stockholm office.', 'responsibilities': 'Design and operate data infrastructure (streaming and batch) an

In [7]:
print("LLM says:   ", ad_info.words_in_article)
print("Actually is:", len(ads["ads4"].split()))

LLM says:    406
Actually is: 449


## 1. Product extractor

In this task you will based on product descriptions, be able to extract out relevant information into structured format using pydantic model together with PydanticAI agent.

a) Read the `products.json` and store all of its descriptions into a list

b) Loop through this list and extract structured output from each of the descriptions. The structure should have the fields: name, price, category, in_stock and description

c) Now try to validate programmatically if the output fields are correct.

> [!NOTE]
> some of the fields might be straightforward as some others might require the use of another LLM agent
